In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

/home/pia/PycharmProjects/spatial_extraction_fairytales/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Exermine Roberta Structure

In [2]:
# model: https://huggingface.co/FacebookAI/xlm-roberta-large-finetuned-conll03-german
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large-finetuned-conll03-german")
model = AutoModelForTokenClassification.from_pretrained("FacebookAI/xlm-roberta-large-finetuned-conll03-german", device_map="auto")

print(tokenizer.cls_token) # <s>
print(tokenizer.sep_token) # </s>
print(tokenizer.mask_token) # <mask>
print(tokenizer.pad_token) # <pad>

Some weights of the model checkpoint at FacebookAI/xlm-roberta-large-finetuned-conll03-german were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


<s>
</s>
<mask>
<pad>


In [3]:
text = "Zur Zeit, wo das Wünschen noch geholfen hat, ward ein Königssohn von einer alten Hexe verwünscht, daß er im Walde in einem großen Eisenofen sitzen sollte. Da brachte er viele Jahre zu, und konnte ihn niemand erlösen."

tokens = tokenizer.tokenize(text)
print(tokens)

['▁Zur', '▁Zeit', ',', '▁wo', '▁das', '▁Wünsche', 'n', '▁noch', '▁geholfen', '▁hat', ',', '▁', 'ward', '▁ein', '▁König', 'sso', 'h', 'n', '▁von', '▁einer', '▁alten', '▁He', 'xe', '▁ver', 'w', 'ün', 's', 'cht', ',', '▁daß', '▁er', '▁im', '▁Wald', 'e', '▁in', '▁einem', '▁großen', '▁Eisen', 'of', 'en', '▁si', 'tzen', '▁sollte', '.', '▁Da', '▁brachte', '▁er', '▁viele', '▁Jahre', '▁zu', ',', '▁und', '▁konnte', '▁ihn', '▁niemand', '▁er', 'lös', 'en', '.']


In [4]:
input_ids = tokenizer(text)["input_ids"]
print(input_ids)

print(tokenizer.convert_ids_to_tokens(input_ids))

[0, 47426, 5502, 4, 4323, 381, 92659, 19, 1838, 226204, 1256, 4, 6, 19364, 599, 89537, 10798, 127, 19, 542, 2235, 83832, 1529, 4878, 493, 434, 7640, 7, 5204, 4, 44124, 72, 566, 59757, 13, 23, 2315, 30909, 138745, 4390, 33, 78, 2731, 12961, 5, 930, 201178, 72, 9510, 12241, 404, 4, 165, 25237, 17889, 46203, 72, 28739, 33, 5, 2]
['<s>', '▁Zur', '▁Zeit', ',', '▁wo', '▁das', '▁Wünsche', 'n', '▁noch', '▁geholfen', '▁hat', ',', '▁', 'ward', '▁ein', '▁König', 'sso', 'h', 'n', '▁von', '▁einer', '▁alten', '▁He', 'xe', '▁ver', 'w', 'ün', 's', 'cht', ',', '▁daß', '▁er', '▁im', '▁Wald', 'e', '▁in', '▁einem', '▁großen', '▁Eisen', 'of', 'en', '▁si', 'tzen', '▁sollte', '.', '▁Da', '▁brachte', '▁er', '▁viele', '▁Jahre', '▁zu', ',', '▁und', '▁konnte', '▁ihn', '▁niemand', '▁er', 'lös', 'en', '.', '</s>']


ID = 0 ->    < s > Token   (Start Eingabe)

ID = 2 ->    < /s > Token   (Ende Eingabe)

In [5]:
print(type(tokenizer))
print(type(tokenizer.backend_tokenizer))
print(tokenizer.backend_tokenizer.model)
print(type(tokenizer.backend_tokenizer.model))

<class 'transformers.models.xlm_roberta.tokenization_xlm_roberta_fast.XLMRobertaTokenizerFast'>
<class 'tokenizers.Tokenizer'>
Unigram(unk_id=3, vocab=[("<s>", 0), ("<pad>", 0), ("</s>", 0), ("<unk>", 0), (",", -3.4635426998138428), ...], byte_fallback=False)
<class 'tokenizers.models.Unigram'>


Model uses Sentence-piece Unigram tokenizer algorithm: https://huggingface.co/docs/transformers/main/en/tokenizer_summary#unigram


## Testing Fairytale

In [50]:
from dataset_preprocessing import load_dataset

In [51]:
DATA_PATH = "data/grimm_fairytales_annotated/fuzzy_loc_3.7.26.json"
dataset = load_dataset(DATA_PATH)

fairytale = next((ft for ft in dataset.fairytales if ft.khm == 1), None)

In [52]:
# Tokenizer with max 512 length chunks and stride

encoding = tokenizer(
    fairytale.text,
    max_length=512,
    truncation=True,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

In [53]:
# encoding.tokens()
# encoding["offset_mapping"]

### Checking if Offset Chunks still fit

In [54]:
# Count Chunks

print("Originaltext:", len(fairytale.text))
print("Anzahl Chunks:", len(encoding["input_ids"]))

for i, offsets in enumerate(encoding["offset_mapping"]):
    print(f"Chunk {i}: {len(offsets)} Tokens")

Originaltext: 7437
Anzahl Chunks: 5
Chunk 0: 512 Tokens
Chunk 1: 512 Tokens
Chunk 2: 512 Tokens
Chunk 3: 512 Tokens
Chunk 4: 493 Tokens


In [55]:
# Offset Chunks

for i, offsets in enumerate(encoding["offset_mapping"]):
    print(f"\n--- Chunk {i} ---")

    for token, offset in zip(
        tokenizer.convert_ids_to_tokens(encoding["input_ids"][i]),
        offsets
    ):
        print(f"{token:20} {offset}")


--- Chunk 0 ---
<s>                  (0, 0)
▁In                  (0, 2)
▁den                 (3, 6)
▁alten               (7, 12)
▁Zeiten              (13, 19)
,                    (19, 20)
▁wo                  (21, 23)
▁das                 (24, 27)
▁Wünsche             (28, 35)
n                    (35, 36)
▁noch                (37, 41)
▁geholfen            (42, 50)
▁hat                 (51, 54)
,                    (54, 55)
▁lebt                (56, 60)
e                    (60, 61)
▁ein                 (62, 65)
▁König               (66, 71)
,                    (71, 72)
▁dessen              (73, 79)
▁Tö                  (80, 82)
chter                (82, 87)
▁waren               (88, 93)
▁alle                (94, 98)
▁schön               (99, 104)
,                    (104, 105)
▁aber                (106, 110)
▁die                 (111, 114)
▁jüngst              (115, 121)
e                    (121, 122)
▁war                 (123, 126)
▁so                  (127, 129)
▁schön         

In [56]:
# Are Token-Offset and token fitting?

for chunk_idx, offsets in enumerate(encoding["offset_mapping"]):

    for start, end in offsets:

        if start == end:
            continue

        assert 0 <= start < end <= len(fairytale.text), (
            f"Invalid offset: {(start, end)}"
        )

In [57]:
for chunk_idx, offsets in enumerate(encoding["offset_mapping"]):

    valid_offsets = [
        (start, end)
        for start, end in offsets
        if start != end
    ]

    chunk_start = min(start for start, end in valid_offsets)
    chunk_end = max(end for start, end in valid_offsets)

    print(
        f"Chunk {chunk_idx}: "
        f"characters {chunk_start}–{chunk_end}"
    )

Chunk 0: characters 0–1889
Chunk 1: characters 1444–3288
Chunk 2: characters 2809–4661
Chunk 3: characters 4210–6052
Chunk 4: characters 5592–7437


In [58]:
covered = set()

for offsets in encoding["offset_mapping"]:

    for start, end in offsets:

        if start == end:
            continue

        covered.update(range(start, end))

missing = [
    i
    for i in range(len(fairytale.text))
    if i not in covered
]

print("Nicht abgedeckte Zeichen:", len(missing))

Nicht abgedeckte Zeichen: 1325


-> Zeichen vermutlich whitespaces, Satzzeichen etc.

In [59]:
# Check Annotation in Token-Offset

def check_annotation_coverage(fairytale, encoding):
    for span in fairytale.resolve_nested_spans():

        covered_tokens = []

        for chunk_idx, offsets in enumerate(encoding["offset_mapping"]):

            for token_idx, (start, end) in enumerate(offsets):

                # Special Tokens ignorieren
                if start == end:
                    continue

                # Token überlappt Annotation
                if start < span.end and end > span.start:
                    covered_tokens.append(
                        (chunk_idx, token_idx, start, end)
                    )

        covered_start = min(start for _, _, start, _ in covered_tokens)
        covered_end = max(end for _, _, _, end in covered_tokens)

        if covered_start > span.start or covered_end < span.end:
            print(
                f"NICHT vollständig abgedeckt:\n"
                f"  Span: {span.text!r}\n"
                f"  Offset: ({span.start}, {span.end})\n"
                f"  Token-Abdeckung: ({covered_start}, {covered_end})"
            )

In [60]:
check_annotation_coverage(
    fairytale,
    encoding
)

In [61]:
def check_uncovered_chars(text, offsets):
    covered = set()

    for chunk_offsets in offsets:
        for start, end in chunk_offsets:
            if start == end:
                continue

            covered.update(range(start, end))

    uncovered = [
        (i, text[i])
        for i in range(len(text))
        if i not in covered
    ]

    print(f"Nicht abgedeckte Zeichen: {len(uncovered)}")

    non_whitespace = [
        (i, char)
        for i, char in uncovered
        if not char.isspace()
    ]

    print(f"Davon Nicht-Whitespace: {len(non_whitespace)}")

    #if non_whitespace:
    #    print("Beispiele:")
    #    print(non_whitespace[:50])

    return uncovered

In [62]:
check_uncovered_chars(
    fairytale.text,
    encoding["offset_mapping"]
)

Nicht abgedeckte Zeichen: 1325
Davon Nicht-Whitespace: 0


[(2, ' '),
 (6, ' '),
 (12, ' '),
 (20, ' '),
 (23, ' '),
 (27, ' '),
 (36, ' '),
 (41, ' '),
 (50, ' '),
 (55, ' '),
 (61, ' '),
 (65, ' '),
 (72, ' '),
 (79, ' '),
 (87, ' '),
 (93, ' '),
 (98, ' '),
 (105, ' '),
 (110, ' '),
 (114, ' '),
 (122, ' '),
 (126, ' '),
 (129, ' '),
 (136, ' '),
 (140, ' '),
 (144, ' '),
 (150, ' '),
 (158, ' '),
 (162, ' '),
 (167, ' '),
 (170, ' '),
 (177, ' '),
 (185, ' '),
 (190, ' '),
 (195, ' '),
 (207, ' '),
 (210, ' '),
 (214, ' '),
 (218, ' '),
 (222, ' '),
 (226, ' '),
 (234, ' '),
 (242, ' '),
 (247, ' '),
 (251, ' '),
 (255, ' '),
 (264, ' '),
 (268, ' '),
 (275, ' '),
 (279, ' '),
 (283, ' '),
 (290, ' '),
 (298, ' '),
 (304, ' '),
 (308, ' '),
 (311, ' '),
 (315, ' '),
 (321, ' '),
 (327, ' '),
 (333, ' '),
 (339, ' '),
 (345, ' '),
 (349, ' '),
 (353, ' '),
 (362, ' '),
 (367, ' '),
 (371, ' '),
 (375, ' '),
 (379, ' '),
 (385, ' '),
 (390, ' '),
 (395, ' '),
 (398, ' '),
 (403, ' '),
 (407, ' '),
 (418, ' '),
 (425, ' '),
 (428, ' '),
 (432

In [63]:
for fairytale in dataset.fairytales:
    print(f"\n{'='*60}")
    print(f"KHM {fairytale.khm}: {fairytale.title}")
    print(f"{'='*60}")

        # Tokenize
    encoding = tokenizer(
        fairytale.text,
        max_length=512,
        truncation=True,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

    check_uncovered_chars(
    fairytale.text,
    encoding["offset_mapping"]
)


KHM 1: Der Froschkönig oder der eiserne Heinrich
Nicht abgedeckte Zeichen: 1325
Davon Nicht-Whitespace: 0

KHM 2: Katze und Maus in Gesellschaft
Nicht abgedeckte Zeichen: 899
Davon Nicht-Whitespace: 0

KHM 3: Marienkind
Nicht abgedeckte Zeichen: 1791
Davon Nicht-Whitespace: 0

KHM 4: Mährchen von einem, der auszog das Fürchten zu lernen
Nicht abgedeckte Zeichen: 3444
Davon Nicht-Whitespace: 0

KHM 5: Der Wolf und die sieben jungen Geislein
Nicht abgedeckte Zeichen: 1032
Davon Nicht-Whitespace: 0

KHM 6: Der treue Johannes
Nicht abgedeckte Zeichen: 2854
Davon Nicht-Whitespace: 0

KHM 7: Der gute Handel
Nicht abgedeckte Zeichen: 1537
Davon Nicht-Whitespace: 0

KHM 8: Der wunderliche Spielmann
Nicht abgedeckte Zeichen: 996
Davon Nicht-Whitespace: 0

KHM 9: Die zwölf Brüder
Nicht abgedeckte Zeichen: 1860
Davon Nicht-Whitespace: 0

KHM 10: Das Lumpengesindel
Nicht abgedeckte Zeichen: 725
Davon Nicht-Whitespace: 0

KHM 11: Brüderchen und Schwesterchen
Nicht abgedeckte Zeichen: 2296
Davon Ni